# Data Cleaning
___
## Objective

The objective of this notebook is to clean and standardize the data based on
the issues identified during data profiling. The cleaning process focuses on
handling missing values, correcting inconsistent data, validating relationships,
and preparing reliable analytical tables for EDA and business analysis.

___

## 1. Load Raw Data

Load the raw tables from PostgreSQL using SQLAlchemy for data cleaning.

In [ ]:
import numpy as np
import pandas as pd
from sqlalchemy import text
from helpers import get_connection, convert_to_hours

In [6]:
engine = get_connection()

In [7]:
user_raw = pd.read_sql_table('users', engine)
teams_raw = pd.read_sql_table('teams', engine)
projects_raw = pd.read_sql_table('projects', engine)
projects_computed_raw = pd.read_sql_table('projects_computed', engine)
tasks_raw = pd.read_sql_table('tasks', engine)
tasks_computed_raw = pd.read_sql_table('tasks_computed', engine)
declarations_raw = pd.read_sql_table('declarations', engine)
subscriptions_raw = pd.read_sql_table('subscriptions', engine)


___

## 2. Define Cleaning Rules

Based on the data profiling results, the following cleaning rules will be applied.

### users
- `deleted_at`: NULL values represent active users. Retain NULL values.

### teams
- `deleted_at`: NULL values represent active teams. Retain NULL values.

### tasks
- `parent_id`: Investigate NULL values and identify the relationship with other IDs before deciding how to handle them.
- `is_container`: Convert `t/f` values to Boolean `True/False`.

### tasks_computed
- `parent_id`: Investigate NULL values and identify the relationship with other IDs before deciding how to handle them.
- `planned_duration`: Convert from string to an appropriate numeric/duration type.
- `elapsed_duration`: Convert from string to an appropriate numeric/duration type.

### declarations
- `date`: Years from `0001` are invalid. Validate and handle dates outside the expected range of 2013–2030.

### subscriptions
- `subscribe_at`: NULL values require investigation before deciding on a cleaning strategy.
- `disabled_at`: NULL values require investigation before deciding on a cleaning strategy.
  

___

## 3. Data Correction

Apply the identified corrections to standardize inconsistent values and convert
columns into appropriate formats for analysis.

In [8]:
tasks_raw['is_container'] = tasks_raw['is_container'].map({
    't': 'True',
    'f': 'False'
})

In [9]:
tasks_computed_raw['planned_duration'] = tasks_computed_raw['planned_duration'].apply(convert_to_hours)
tasks_computed_raw['elapsed_duration'] = tasks_computed_raw['elapsed_duration'].apply(convert_to_hours)

In [10]:
tasks_computed_raw[['planned_duration', 'elapsed_duration']]

,planned_duration,elapsed_duration
0,0.0,0.0
1,0.0,3.5
2,0.0,6.0
3,0.0,0.0
4,0.0,2.0
...,...,...
628461,3.0,3.0
628462,5.0,5.0
628463,0.0,0.0
628464,6.0,0.0


___

## 4. Handle Missing Values

Handle missing values based on the findings from data profiling and the relationships identified during the cleaning process. NULL values that have
a valid business or structural meaning are retained.

### tasks / tasks_computed — parent_id

- `parent_id` contains NULL values representing top-level tasks, so they are retained.
- Non-null `parent_id` values should match an existing `id` within the respective table.
- Both tables use `parent_id` as a self-referencing key to represent hierarchical task relationships.
- No correction is required for valid `parent_id` values.

In [11]:
parent_ids = tasks_raw["parent_id"].dropna().unique()
ids = tasks_raw["id"].dropna().unique()

matches = set(parent_ids) & set(ids)
match_percentage = len(matches) / len(parent_ids) * 100

print("Parent IDs:", len(parent_ids))
print("Matching IDs:", len(matches))
print(f"Match percentage: {match_percentage:.2f}%")

Parent IDs: 68242
Matching IDs: 68242
Match percentage: 100.00%


In [12]:
parent_ids = tasks_computed_raw["parent_id"].dropna().unique()
ids = tasks_raw["id"].dropna().unique()

matches = set(parent_ids) & set(ids)
match_percentage = len(matches) / len(parent_ids) * 100

print("Parent IDs:", len(parent_ids))
print("Matching IDs:", len(matches))
print(f"Match percentage: {match_percentage:.2f}%")

Parent IDs: 58536
Matching IDs: 58536
Match percentage: 100.00%


### declarations — date

- The actual valid date range is **2013–2030**. Dates with the year **0001** are misentered values.
- When loaded into pandas, Python treats the year `0001` values as `NaT` because `datetime64[ns]` cannot represent year `0001`.
- The misentered `0001` dates will be removed during cleaning.

In [13]:
declarations_raw["date"] = pd.to_datetime(declarations_raw["date"], errors="coerce")

pd.DataFrame({
    "year": declarations_raw["date"].sort_values(ascending=True).dt.year.unique()
})

,year
0,2013.0
1,2014.0
2,2015.0
3,2016.0
4,2017.0
5,2018.0
6,2019.0
7,2020.0
8,2021.0
9,2022.0


In [14]:
declarations_raw = declarations_raw.dropna(subset=["date"])

### subscriptions — subscribe_at / disabled_at

The `status` column has a clear relationship with the `subscribe_at` and
`disabled_at` date columns. Missing values were therefore investigated
based on subscription status before applying any cleaning rules.

In [15]:
subscriptions_raw.groupby("status")[["subscribe_at", "disabled_at"]].apply(
    lambda x: x.isna().sum()
)

,subscribe_at,disabled_at
status,,
active,0,272
disabled,2226,1
pending,0,12
trialing,22,22


In [16]:
subscriptions_raw.groupby("status")[["subscribe_at", "disabled_at"]].apply(
    lambda x: x.isna().mean() * 100
).round(2)

,subscribe_at,disabled_at
status,,
active,0.00,100.00
disabled,87.91,0.04
pending,0.00,100.00
trialing,100.00,100.00


### Anomaly Detection

One record with `status = disabled` has a NULL `disabled_at` value.
This is inconsistent with the other disabled subscriptions, and no
relationship with other columns was found to explain the missing date.
The record is therefore treated as an anomalous record.

In [17]:
subscriptions_raw[
    (subscriptions_raw["status"] == "disabled") &
    (subscriptions_raw["disabled_at"].isna())
]

,team_id,subscribe_at,disabled_at,status,offer,trial_start_at,trial_end_at
1825,99726867-5377-40e2-a614-e1aead83244f,2020-04-01 00:00:00+00:00,NaT,disabled,time,2020-04-01 00:00:00+00:00,2021-04-01 00:00:00+00:00


In [18]:
subscriptions_raw = subscriptions_raw[
    ~(
        (subscriptions_raw["status"] == "disabled") &
        (subscriptions_raw["disabled_at"].isna())
    )
]

### Final Cleaning Decision

- Investigated the relationship between `status`, `subscribe_at`, and `disabled_at`.
- Found that valid `disabled` records can contain both dates, so they are retained.
- NULL values consistent with the subscription status are retained.
- Identified **1 anomalous disabled record** with a NULL `disabled_at` (**1 of 2,532; 0.04%**).
- The anomalous record is removed from the clean dataset; the raw data remains unchanged.

___

## 6. Create Clean Analytical Tables

Create cleaned versions of the raw tables after applying the defined cleaning
rules and corrections. The cleaned tables will be stored separately from the
raw data for downstream EDA and business analysis.

In [19]:
users_cleaned = user_raw.copy()
teams_cleaned = teams_raw.copy()
projects_cleaned = projects_raw.copy()
projects_computed_cleaned = projects_computed_raw.copy()
tasks_cleaned = tasks_raw.copy()
tasks_computed_cleaned = tasks_computed_raw.copy()
declarations_cleaned = declarations_raw.copy()
subscriptions_cleaned = subscriptions_raw.copy()


In [20]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS cleaned"))

In [21]:
users_cleaned.to_sql('users',engine,schema='cleaned',if_exists='replace',index=False)
teams_cleaned.to_sql('teams',engine,schema='cleaned',if_exists='replace',index=False)
projects_cleaned.to_sql('projects',engine,schema='cleaned',if_exists='replace',index=False)
projects_computed_cleaned.to_sql('projects_computed',engine,schema='cleaned',if_exists='replace',index=False)
tasks_cleaned.to_sql('tasks',engine,schema='cleaned',if_exists='replace',index=False)
tasks_computed_cleaned.to_sql('tasks_computed',engine,schema='cleaned',if_exists='replace',index=False)
declarations_cleaned.to_sql('declarations',engine,schema='cleaned',if_exists='replace',index=False)
subscriptions_cleaned.to_sql('subscriptions',engine,schema='cleaned',if_exists='replace',index=False)

837

___

## 7. Cleaning Summary

- `users` and `teams`: NULL `deleted_at` values were retained as they represent active records.
- `tasks` and `tasks_computed`: NULL `parent_id` values were retained as they represent top-level tasks. Non-null values were validated as self-referencing IDs.
- `tasks`: `is_container` values were standardized to Boolean values.
- `tasks_computed`: `planned_duration` and `elapsed_duration` were converted to numeric hours.
- `declarations`: Invalid year `0001` dates were identified and removed; the valid date range is 2013–2030.
- `subscriptions`: NULL date values consistent with subscription status were retained, while one anomalous disabled record with a missing `disabled_at` was removed.
- No duplicate records or outliers required treatment.
- Cleaned tables were created separately in the `cleaned` PostgreSQL schema for downstream EDA and business analysis.